# Notebook 3: FRIP Signals and Drive Exports

Loads the pre-materialized NPP and Covariate stacks from Notebook 1 and the pre-aggregated GEDI MODIS-scale asset from Notebook 2.
It concatenates them instantly in one millisecond via `ee.Image.cat`, computes the multi-scale Flooding Role in Productivity (FRIP) and its Mann-Kendall trend, and exports all 42 analysis stack GeoTIFFs to Google Drive.

### Output GeoTIFFs:
1. **40 Multi-scale Stacks** (20 scales [5km to 100km] x 2 basins):
   - Standardized at EPSG:4326 geographic resolution.
   - Contains **12 bands** including cross-sectional FRIP, the GEE-computed **FRIP Mann-Kendall trend (frip_mk_tau)**, GEDI structural signals, and environmental covariates.
2. **2 Native-scale Stacks** (1 per basin):
   - Standardized at native MODIS resolution (~463m, EPSG:4326).
   - Contains **11 bands** (GEDI signals + environmental covariates + NPP median) for high-resolution spatial modeling *without* FRIP.

### Band Structure (11 bands total for multi-scale):
- **`frip`** (1 band): Cross-sectional Spearman correlation
- **`frip_mk_tau`** (1 band): Mann-Kendall trend τ of annual FRIP across 2001–2023
- **`uoi`**, **`uoi_sd`**, **`rh98`**, **`gedi_n`** (4 GEDI bands): Openness, height, footprint count (pre-masked with undisturbed forest at 25m scale in Notebook 2)
- **`elevation`**, **`slope`**, **`hnd`**, **`precip`**, **`clay`**, **`forest_fraction`** (6 covariate bands)

In [ ]:
# =============================================================================
# BLOCK 1: SETUP AND CONFIGURATION
# =============================================================================
import ee

try:
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("✓ GEE initialized successfully!")
except Exception as e:
    ee.Authenticate()
    ee.Initialize(project='quantum-bonus-434714-t2')
    print("✓ GEE initialized successfully!")

# Asset paths (from NB1 and NB2)
ASSET_ROOT = 'projects/quantum-bonus-434714-t2/assets/DefaunationFromSpace'

# Study regions (must match NB1 & NB2)
CONGO_BBOX = ee.Geometry.Rectangle([8, -12, 35, 8])
AMAZON_BBOX = ee.Geometry.Rectangle([-73, -18, -44, 8])
BASINS = [('Congo', CONGO_BBOX), ('Amazon', AMAZON_BBOX)]

# Analysis parameters
SCALES = list(range(5000, 105000, 5000))  # 5km to 100km in 5km steps
YEARS = list(range(2001, 2024))

# Masking thresholds
FOREST_COVER_THRESHOLD = 0.95
MAX_ELEVATION = 1000
SLOPE_MAX = 10  # Harmonized with NB1/NB2

# Reference projection (from NB1)
_modis_col = ee.ImageCollection('MODIS/061/MOD17A3HGF').select('Npp')
MODIS_PROJ = ee.Image(_modis_col.first()).projection()
MODIS_SCALE = 463.3127165279165  # MODIS equatorial pixel size in meters

print("✓ Configuration loaded.")
print(f"  Scales: {SCALES[0]/1000:.0f}km - {SCALES[-1]/1000:.0f}km ({len(SCALES)} scales)")
print(f"  Basins: {[b[0] for b in BASINS]}")


In [ ]:
# =============================================================================
# BLOCK 2: IN-MEMORY COMPUTATION LOGIC
# =============================================================================

def compute_frip_mk_tau(annual_frip):
    """Computes pixel-wise Mann-Kendall trend tau across 23 years of annual FRIP.
    
    Tau = S / (n * (n - 1) / 2)
    where S = sum_{i < j} sign(x_j - x_i). For n = 23, total pairs = 253.
    """
    band_names = [f'FRIP_{y}' for y in YEARS]
    signs = []
    
    # Compute sign for all 253 unique year pairs
    for i in range(len(band_names)):
        for j in range(i + 1, len(band_names)):
            img_i = annual_frip.select(band_names[i])
            img_j = annual_frip.select(band_names[j])
            diff = img_j.subtract(img_i)
            # sign: 1 if diff > 0, -1 if diff < 0, 0 if diff == 0
            sign = diff.gt(0).subtract(diff.lt(0)).rename('sign')
            signs.append(sign)
            
    s = ee.ImageCollection.fromImages(signs).sum()
    tau = s.divide(253).rename('frip_mk_tau')
    return tau

def load_base_stack(basin_name):
    """Loads NPP and Covariate stacks from Notebook 1 and GEDI MODIS stack from Notebook 2."""
    npp = ee.Image(f'{ASSET_ROOT}/NppStack_{basin_name}')
    covs = ee.Image(f'{ASSET_ROOT}/CovStack_{basin_name}')
    gedi = ee.Image(f'{ASSET_ROOT}/GediUndisturbedModis_{basin_name}')
    return ee.Image.cat([npp, covs, gedi])

def build_scale_stack(base, basin_name, scale):
    """Computes all signals and covariates in memory and stacks into 11 bands.
    
    Inputs are already at MODIS scale, completely avoiding native-to-MODIS reduceResolution bottlenecks.
    """
    base_proj = base.projection()
    
    # -------------------------------------------------------------------------
    # 1. FRIP computation (Cross-sectional + Annual + Mann-Kendall Trend)
    # -------------------------------------------------------------------------
    # Apply forest cover mask to base
    frip_masked = base.updateMask(
        base.select('forest_fraction').gte(FOREST_COVER_THRESHOLD)
    )
    
    # Cross-sectional FRIP
    frip_cross = frip_masked.select(['flood_freq', 'Npp_median']).setDefaultProjection(base_proj).reduceResolution(
        reducer=ee.Reducer.spearmansCorrelation(),
        maxPixels=65535
    ).reproject(crs='EPSG:4326', scale=scale).select('correlation').rename('frip')
    
    # Apply valid-pixels-count threshold (>10% valid pixels coverage)
    frip_cross = frip_cross.updateMask(frip_cross.mask().gt(0.1))
    
    # Annual FRIP
    def get_annual_corr(year_index):
        year_index = ee.Number(year_index)
        year = ee.Number(2001).add(year_index)
        npp_band = ee.String('NPP_').cat(year.format('%d'))
        
        corr = frip_masked.select([npp_band, 'flood_freq']).setDefaultProjection(base_proj).reduceResolution(
            reducer=ee.Reducer.spearmansCorrelation(),
            maxPixels=65535
        ).reproject(crs='EPSG:4326', scale=scale).select('correlation')
        
        return corr.updateMask(corr.mask().gt(0.1)).set('year', year)
    
    annual_list = ee.List.sequence(0, len(YEARS) - 1).map(get_annual_corr)
    frip_annual = ee.ImageCollection.fromImages(annual_list).toBands()
    band_names = [f'FRIP_{y}' for y in YEARS]
    frip_annual = frip_annual.rename(band_names)
    
    # Compute Mann-Kendall trend tau
    frip_mk_tau = compute_frip_mk_tau(frip_annual)
    
    # -------------------------------------------------------------------------
    # 2. GEDI signals (Masked and Aggregated)
    # -------------------------------------------------------------------------
    # Apply forest cover AND topographic masks to the MODIS-scale GEDI bands
    # GEDI bands ('uoi', 'rh98', 'gedi_n') are pre-masked with JRC undisturbed forest at 25m resolution.
    # Here we apply the additional topo masks and forest cover fraction.
    gedi_masked = base.select(['uoi', 'uoi_sd', 'rh98', 'gedi_n']).updateMask(
        base.select('forest_fraction').gte(FOREST_COVER_THRESHOLD)
        .And(base.select('elevation').lt(MAX_ELEVATION))
        .And(base.select('slope').lt(SLOPE_MAX))
    )
    
    # Aggregate GEDI from MODIS scale to target analysis scale
    uoi_agg = gedi_masked.select('uoi').setDefaultProjection(base_proj).reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535
    ).reproject(crs='EPSG:4326', scale=scale).rename('uoi')
    
    rh98_agg = gedi_masked.select('rh98').setDefaultProjection(base_proj).reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535
    ).reproject(crs='EPSG:4326', scale=scale).rename('rh98')
    
    n_agg = gedi_masked.select('gedi_n').setDefaultProjection(base_proj).reduceResolution(
        reducer=ee.Reducer.sum(), maxPixels=65535
    ).reproject(crs='EPSG:4326', scale=scale).rename('gedi_n')
    
    uoi_sd_agg = gedi_masked.select('uoi_sd').setDefaultProjection(base_proj).reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535
    ).reproject(crs='EPSG:4326', scale=scale).rename('uoi_sd')
    
    # -------------------------------------------------------------------------
    # 3. Covariates (Aggregated)
    # -------------------------------------------------------------------------
    covariates = ['elevation', 'slope', 'hnd', 'precip', 'clay', 'forest_fraction']
    covs_agg = base.select(covariates).setDefaultProjection(base_proj).reduceResolution(
        reducer=ee.Reducer.mean(), maxPixels=65535
    ).reproject(crs='EPSG:4326', scale=scale)
    
    # -------------------------------------------------------------------------
    # 4. Concatenate into a single 12-band stack
    # -------------------------------------------------------------------------
    stack = ee.Image.cat([
        frip_cross,    # 1 band
        frip_mk_tau,   # 1 band
        uoi_agg,       # 1 band
        uoi_sd_agg,    # 1 band
        rh98_agg,      # 1 band
        n_agg,         # 1 band
        covs_agg       # 6 bands
    ]).toFloat()
    
    return stack

def build_native_stack(base):
    """Assembles high-res GEDI + covariates at native MODIS scale (~463m) without FRIP.
    
    Used for pixel-level spatial modelling of GEDI indicators at native resolution.
    """
    # Apply topo + forest cover fraction thresholds to the GEDI bands
    gedi_masked = base.select(['uoi', 'uoi_sd', 'rh98', 'gedi_n']).updateMask(
        base.select('forest_fraction').gte(FOREST_COVER_THRESHOLD)
        .And(base.select('elevation').lt(MAX_ELEVATION))
        .And(base.select('slope').lt(SLOPE_MAX))
    )
    
    covs = base.select(['elevation', 'slope', 'hnd', 'precip', 'clay', 'forest_fraction', 'Npp_median'])
    
    stack = ee.Image.cat([
        gedi_masked,
        covs
    ]).toFloat()
    
    return stack

print("✓ In-memory computation logic loaded.")


In [ ]:
def run_unit_tests():
    print("Running unit tests (using Congo basin)...\n")
    passed = 0
    failed = 0
    
    # --- Test 1: load_base_stack ---
    try:
        print("  [1/5] Loading base stack assets...")
        base = load_base_stack('Congo')
        assert isinstance(base, ee.Image), "Base stack must be an ee.Image"
        base_bands = base.bandNames().getInfo()
        assert 'Npp_median' in base_bands, "Missing Npp_median band"
        assert 'uoi' in base_bands, "Missing uoi band"
        assert 'clay' in base_bands, "Missing clay band"
        passed += 1
        print(f"    ✓ Base stack loaded successfully. Total bands: {len(base_bands)}")
    except Exception as e:
        failed += 1
        print(f"    ✗ load_base_stack FAILED: {e}")
        print("      (Note: This might be expected if downstream assets have not been exported yet.)")
        
    # --- Test 2: build_scale_stack band count ---
    try:
        print("  [2/5] Validating multi-scale stack band count (Congo at 50km)...")
        base = load_base_stack('Congo')
        stack = build_scale_stack(base, 'Congo', 50000)
        assert isinstance(stack, ee.Image), "Scale stack must return an ee.Image"
        bands = stack.bandNames().getInfo()
        assert len(bands) == 12, f"Expected 12 bands, got {len(bands)}: {bands}"
        passed += 1
        print(f"    ✓ Correct multi-scale band count: 12 bands assembled")
    except Exception as e:
        failed += 1
        print(f"    ✗ build_scale_stack band count FAILED: {e}")
        
    # --- Test 3: build_scale_stack band names and structure ---
    try:
        print("  [3/5] Validating multi-scale stack band names...")
        base = load_base_stack('Congo')
        stack = build_scale_stack(base, 'Congo', 50000)
        bands = stack.bandNames().getInfo()
        required = ['frip', 'frip_mk_tau', 'uoi', 'uoi_sd', 'rh98', 'gedi_n',
                    'elevation', 'slope', 'hnd', 'precip', 'clay', 'forest_fraction']
        missing = [b for b in required if b not in bands]
        assert not missing, f"Missing required bands: {missing}"
        passed += 1
        print("    ✓ All 12 required signals and covariates present (including frip_mk_tau)")
    except Exception as e:
        failed += 1
        print(f"    ✗ build_scale_stack band names FAILED: {e}")
        
    # --- Test 4: build_scale_stack graph evaluation (forces lazy evaluation) ---
    try:
        print("  [4/5] Verifying multi-scale stack graph evaluation (forces lazy evaluation)...\n" 
              "        (This triggers server-side validation of image collections and castings)")
        base = load_base_stack('Congo')
        stack = build_scale_stack(base, 'Congo', 50000)
        # Evaluate centroid to force execution of the stack logic (including MK trend collection)
        sample_geom = base.geometry().centroid()
        sample_val = stack.reduceRegion(
            reducer=ee.Reducer.first(),
            geometry=sample_geom,
            scale=50000
        ).getInfo()
        assert sample_val is not None, "Evaluation returned None"
        passed += 1
        print(f"    ✓ Graph evaluation successful. Sample bands validated: {sorted(list(sample_val.keys()))}")
    except Exception as e:
        failed += 1
        print(f"    ✗ Graph evaluation FAILED: {e}")
        
    # --- Test 5: build_native_stack band count and structure ---
    try:
        print("  [5/5] Validating native-scale stack...")
        base = load_base_stack('Congo')
        native_stack = build_native_stack(base)
        assert isinstance(native_stack, ee.Image), "Native stack must return an ee.Image"
        native_bands = native_stack.bandNames().getInfo()
        assert len(native_bands) == 11, f"Expected 11 bands, got {len(native_bands)}: {native_bands}"
        required_native = ['uoi', 'uoi_sd', 'rh98', 'gedi_n', 'elevation', 'slope', 'hnd', 'precip', 'clay', 'forest_fraction', 'Npp_median']
        missing_native = [b for b in required_native if b not in native_bands]
        assert not missing_native, f"Missing native bands: {missing_native}"
        passed += 1
        print("    ✓ Correct native-scale band count and structure: 11 bands assembled")
    except Exception as e:
        failed += 1
        print(f"    ✗ build_native_stack FAILED: {e}")
        
    total = passed + failed
    print(f"\n{'='*60}")
    if failed == 0:
        print(f"  ✓ ALL {passed}/{total} TESTS PASSED SUCCESSFULLY!")
        print("  Ready to export multi-scale analysis stacks.")
    else:
        print(f"  ✗ {passed}/{total} passed, {failed} failed. Fix failures or ensure Stage 1 and Stage 2 assets exist.")
    print(f"{'='*60}")

run_unit_tests()


In [ ]:
# =============================================================================
# BLOCK 4: EXPORT TO DRIVE
# =============================================================================

def safe_start(task, asset_id):
    """Deletes existing asset (if any) then starts export task.
    Prevents 'Asset already exists' failures on re-runs."""
    try:
        ee.data.deleteAsset(asset_id)
        print(f"    Deleted existing: {asset_id.split('/')[-1]}")
    except Exception:
        pass
    task.start()

def export_all_datasets(dry_run=True):
    """Launches exports for all 42 GeoTIFFs to Google Drive.
    
    - 40 Multi-scale stacks (20 scales x 2 basins)
    - 2 Native-scale stacks (1 per basin)
    Saves to Drive folder: 'DefaunationSynthesis/AnalysisStack/'
    """
    tasks = []
    
    for basin_name, basin_geom in BASINS:
        try:
            base = load_base_stack(basin_name)
        except Exception as e:
            print(f"✗ Error loading base stack for {basin_name}: {e}")
            print("  Ensure that Notebook 1 and Notebook 2 exports have completed successfully first!")
            continue
        
        # 1. Configure the 20 multi-scale exports (11 bands)
        for scale in SCALES:
            stack = build_scale_stack(base, basin_name, scale)
            
            task = ee.batch.Export.image.toDrive(
                image=stack,
                description=f'analysis_stack_{scale}_{basin_name}',
                folder='DefaunationSynthesis/AnalysisStack',
                fileNamePrefix=f'analysis_stack_{scale}_{basin_name}',
                region=basin_geom,
                scale=scale,
                crs='EPSG:4326',
                maxPixels=1e13
            )
            tasks.append((task, f'analysis_stack_{scale}_{basin_name}'))
            
        # 2. Configure the 1 native-scale export (10 bands)
        native_stack = build_native_stack(base)
        task_native = ee.batch.Export.image.toDrive(
            image=native_stack,
            description=f'analysis_stack_native_{basin_name}',
            folder='DefaunationSynthesis/AnalysisStack',
            fileNamePrefix=f'analysis_stack_native_{basin_name}',
            region=basin_geom,
            scale=MODIS_SCALE, # Native scale (~463m)
            crs='EPSG:4326',   # standard geodetic
            maxPixels=1e13
        )
        tasks.append((task_native, f'analysis_stack_native_{basin_name}'))
            
    print(f"✓ {len(tasks)} Drive export tasks configured:")
    print(f"  - 40 multi-scale exports (11 bands each)")
    print(f"  - 2 native-scale exports (10 bands each)")
    
    if dry_run:
        print("\nDRY RUN. Call export_all_datasets(dry_run=False) to launch.")
    else:
        for task, name in tasks:
            task.start()
            print(f"  ✓ Started Drive export: {name}")
        print("\n✓ All 42 Drive exports started!")
        print("  Monitor at: https://code.earthengine.google.com/tasks")

export_all_datasets(dry_run=True)
